In [ ]:

# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D9 — Branch B — OCR-based Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed corrected Stage 1 document-grounded reference dataset
# - Branch B parsed extraction
# - Branch B technical diagnostics
# - D9 comparison rules frozen from final Validation A
# ============================================================

from google.colab import files
from pathlib import Path
from difflib import SequenceMatcher

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D9"

DOCUMENT_NAME = (
    "Estatística do Movimento Fisiológico da População "
    "de Portugal — Ano de 1925"
)

BRANCH = "B"
BRANCH_NAME = "OCR-based structural conversion"

EXPECTED_RECORD_COUNT = 19

EXPECTED_CATEGORY_COUNTS = {
    "Publication metadata": 5,
    "Index entry": 6,
    "Statistical value": 5,
    "Document structure": 3,
}

FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location",
]

ALIGNMENT_IDENTITY_FIELDS = [
    "Category",
    "Topic",
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Topic",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location",
]

DESCRIPTION_DIAGNOSTIC_FIELD = "Description"

OUTPUT_DIR = Path("outputs_D9_validation_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH, "-", BRANCH_NAME)
print("Expected reference records:", EXPECTED_RECORD_COUNT)
print("Alignment identity fields:", ALIGNMENT_IDENTITY_FIELDS)
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)


In [ ]:

# ============================================================
# 2. Upload validation inputs
# ============================================================
# Required:
#   1) D9_reference_values.csv
#   2) D9_branch_B_parsed_extraction.json
#   3) D9_branch_B_technical_diagnostics.json

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [
    name for name in uploaded_files
    if name.lower().endswith(".csv")
]

json_files = [
    name for name in uploaded_files
    if name.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one D9 Stage 1 reference CSV."
    )

if len(json_files) != 2:
    raise ValueError(
        "Upload exactly two JSON files: the Branch B parsed "
        "extraction and technical diagnostics."
    )

REFERENCE_FILE = csv_files[0]
PARSED_EXTRACTION_FILE = None
TECHNICAL_DIAGNOSTICS_FILE = None

for file_name in json_files:

    with open(file_name, "r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(obj.get("records"), list)
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structurally_evaluable" in obj
        and "valid_json" in obj
    ):
        TECHNICAL_DIAGNOSTICS_FILE = file_name

if PARSED_EXTRACTION_FILE is None:
    raise ValueError(
        "Could not identify the D9 Branch B parsed extraction."
    )

if TECHNICAL_DIAGNOSTICS_FILE is None:
    raise ValueError(
        "Could not identify the D9 Branch B technical diagnostics."
    )

print("Reference:", REFERENCE_FILE)
print("Parsed extraction:", PARSED_EXTRACTION_FILE)
print("Technical diagnostics:", TECHNICAL_DIAGNOSTICS_FILE)


In [ ]:

# ============================================================
# 3. Load inputs and verify identity/provenance
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_FILE,
    dtype=object,
    keep_default_na=False,
    encoding="utf-8-sig",
)

def restore_null(value):
    return None if value == "" else value

for column in reference_df.columns:
    reference_df[column] = reference_df[column].map(restore_null)

with open(
    PARSED_EXTRACTION_FILE,
    "r",
    encoding="utf-8-sig",
) as f:
    extraction_json = json.load(f)

with open(
    TECHNICAL_DIAGNOSTICS_FILE,
    "r",
    encoding="utf-8-sig",
) as f:
    technical_diagnostics = json.load(f)

for artefact_name, artefact in {
    "parsed extraction": extraction_json,
    "technical diagnostics": technical_diagnostics,
}.items():

    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id: "
            f"{artefact.get('document_id')}"
        )

    if artefact.get("branch") != BRANCH:
        raise ValueError(
            f"Unexpected {artefact_name} branch: "
            f"{artefact.get('branch')}"
        )

extracted_records = extraction_json["records"]
extracted_df = pd.DataFrame(extracted_records)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "parsed_extraction_file": PARSED_EXTRACTION_FILE,
    "parsed_extraction_sha256": sha256_file(PARSED_EXTRACTION_FILE),
    "technical_diagnostics_file": TECHNICAL_DIAGNOSTICS_FILE,
    "technical_diagnostics_sha256":
        sha256_file(TECHNICAL_DIAGNOSTICS_FILE),
}

print("Reference shape:", reference_df.shape)
print("Extraction shape:", extracted_df.shape)


In [ ]:

# ============================================================
# 4. Import Branch B technical/schema status
# ============================================================

structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False,
    )
)

record_schema_valid = bool(
    technical_diagnostics.get(
        "record_schema_valid",
        False,
    )
)

field_types_valid = bool(
    technical_diagnostics.get(
        "field_types_valid",
        False,
    )
)

schema_validity = bool(
    structurally_evaluable
    and record_schema_valid
    and field_types_valid
)

schema_diagnostics = {
    "valid_json": bool(
        technical_diagnostics.get("valid_json", False)
    ),
    "record_schema_valid": record_schema_valid,
    "field_types_valid": field_types_valid,
    "structurally_evaluable": structurally_evaluable,
    "schema_validity": schema_validity,
}

if not structurally_evaluable:
    raise ValueError(
        "D9 Branch B output is not structurally evaluable. "
        "Content-level validation cannot proceed."
    )

print(json.dumps(
    schema_diagnostics,
    indent=2,
    ensure_ascii=False,
))


In [ ]:

# ============================================================
# 5. Verify fixed corrected Stage 1 reference and extraction
# ============================================================

if reference_df.columns.tolist() != FIELDS:
    raise ValueError(
        "D9 Stage 1 reference schema does not match the fixed field list."
    )

if len(reference_df) != EXPECTED_RECORD_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_RECORD_COUNT} reference records; "
        f"found {len(reference_df)}."
    )

reference_category_counts = (
    reference_df["Category"].value_counts().to_dict()
)

if reference_category_counts != EXPECTED_CATEGORY_COUNTS:
    raise ValueError(
        "D9 Stage 1 category counts do not match the fixed design."
    )

missing_extraction_fields = [
    field for field in FIELDS
    if field not in extracted_df.columns
]

extraction_cmp = extracted_df.copy(deep=True)

for field in missing_extraction_fields:
    extraction_cmp[field] = None

extraction_cmp = extraction_cmp[FIELDS].copy()
reference_cmp = reference_df[FIELDS].copy(deep=True)

extraction_record_count_valid = (
    len(extraction_cmp) == EXPECTED_RECORD_COUNT
)

extraction_category_counts = (
    extraction_cmp["Category"]
    .value_counts(dropna=False)
    .to_dict()
)

extraction_category_counts_valid = (
    extraction_category_counts == EXPECTED_CATEGORY_COUNTS
)

content_diagnostics = {
    "reference_record_count_valid": True,
    "reference_category_counts_valid": True,
    "extraction_record_count_valid":
        bool(extraction_record_count_valid),
    "extraction_category_counts_valid":
        bool(extraction_category_counts_valid),
    "branch_B_scope_complete":
        technical_diagnostics.get("scope_complete"),
    "branch_B_content_diagnostics":
        technical_diagnostics.get("content_diagnostics"),
}

print("Reference records:", len(reference_cmp))
print("Extracted records:", len(extraction_cmp))
print("Missing extraction columns:", missing_extraction_fields)


In [ ]:
# ============================================================
# 6. Comparison-only normalisation
# ============================================================

def is_missing(value):
    if value is None:
        return True

    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def normalise_text(value):
    if is_missing(value):
        return None

    text = str(value)

    text = "".join(
        character
        for character in text
        if unicodedata.category(character) != "Cf"
    )

    text = unicodedata.normalize("NFKC", text)

    text = (
        text.replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00a0", " ")
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text.casefold()


def identity_text(value):
    text = normalise_text(value)

    if text is None:
        return ""

    text = re.sub(r"[^a-z0-9à-ÿ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def lexical_similarity(first, second):
    from difflib import SequenceMatcher

    first_text = normalise_text(first) or ""
    second_text = normalise_text(second) or ""

    if not first_text and not second_text:
        return 1.0

    if not first_text or not second_text:
        return 0.0

    return SequenceMatcher(
        None,
        first_text,
        second_text,
    ).ratio()


def exact_normalised_text_match(first, second):
    return normalise_text(first) == normalise_text(second)


In [ ]:
# ============================================================
# 7. Conservative controlled comparison rules
# ============================================================

UNIT_EQUIVALENCE_MAP = {
    "year": "year",
    "square kilometres": "square kilometres",
    "square kilometers": "square kilometres",
    "km²": "square kilometres",
    "km2": "square kilometres",
    "people": "people",
    "persons": "people",
    "inhabitants per square kilometre":
        "inhabitants per square kilometre",
    "inhabitants per square kilometer":
        "inhabitants per square kilometre",
    "per thousand inhabitants":
        "per thousand inhabitants",
}


def canonical_unit(value):
    text = normalise_text(value)

    if text is None:
        return None

    return UNIT_EQUIVALENCE_MAP.get(text, text)


def canonical_reporting_period(value):
    text = normalise_text(value)

    if text is None:
        return None

    text = re.sub(r"\s*-\s*", "-", text)

    return text


def canonical_source_location(value):
    text = normalise_text(value)

    if text is None:
        return None

    text = re.sub(r"^physical\s+", "", text)
    text = text.replace("índice", "index")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def unit_correct(reference_value, extracted_value):
    return (
        canonical_unit(reference_value)
        == canonical_unit(extracted_value)
    )


def period_correct(reference_value, extracted_value):
    return (
        canonical_reporting_period(reference_value)
        == canonical_reporting_period(extracted_value)
    )


def source_location_correct(reference_value, extracted_value):
    return (
        canonical_source_location(reference_value)
        == canonical_source_location(extracted_value)
    )


print(
    "Conservative D9 comparison rules loaded. "
    "Final D9 Branch A comparison rules are frozen and reused unchanged; no Branch-B-specific equivalence rules are added."
)


In [ ]:
# ============================================================
# 8. Type-aware Value comparison
# ============================================================

def numeric_value(value):
    if is_missing(value) or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    return None


def value_correct(reference_value, extracted_value):

    if is_missing(reference_value) and is_missing(extracted_value):
        return True

    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if reference_number is not None and extracted_number is not None:
        return math.isclose(
            reference_number,
            extracted_number,
            rel_tol=0.0,
            abs_tol=1e-12,
        )

    if isinstance(reference_value, str) and isinstance(extracted_value, str):
        return (
            normalise_text(reference_value)
            == normalise_text(extracted_value)
        )

    return False


In [ ]:
# ============================================================
# 9. Prepare fixed identity keys
# ============================================================


reference_comparison_df = reference_cmp.copy()
extracted_comparison_df = extraction_cmp.copy()

reference_comparison_df["_reference_index"] = np.arange(
    len(reference_comparison_df)
)

extracted_comparison_df["_extraction_index"] = np.arange(
    len(extracted_comparison_df)
)

for frame in [
    reference_comparison_df,
    extracted_comparison_df,
]:
    frame["_identity_category"] = frame["Category"].map(
        identity_text
    )

    frame["_identity_topic"] = frame["Topic"].map(
        identity_text
    )

    frame["_identity_key"] = list(zip(
        frame["_identity_category"],
        frame["_identity_topic"],
    ))


reference_duplicate_identity_count = int(
    reference_comparison_df["_identity_key"].duplicated().sum()
)

extraction_duplicate_identity_count = int(
    extracted_comparison_df["_identity_key"].duplicated().sum()
)

print(
    "Reference duplicate Category+Topic identities:",
    reference_duplicate_identity_count,
)

print(
    "Extraction duplicate Category+Topic identities:",
    extraction_duplicate_identity_count,
)

if reference_duplicate_identity_count:
    raise AssertionError(
        "The fixed D9 reference does not have unique Category+Topic identities."
    )


In [ ]:
# ============================================================
# 10. One-to-one outcome-independent alignment
# ============================================================

extraction_key_to_indices = {}

for _, row in extracted_comparison_df.iterrows():
    extraction_key_to_indices.setdefault(
        row["_identity_key"],
        []
    ).append(int(row["_extraction_index"]))


matched_pairs = []
matched_extraction_indices = set()
missing_reference_indices = []

for _, reference_row in reference_comparison_df.iterrows():

    reference_index = int(
        reference_row["_reference_index"]
    )

    key = reference_row["_identity_key"]

    available_indices = [
        index
        for index in extraction_key_to_indices.get(key, [])
        if index not in matched_extraction_indices
    ]

    if len(available_indices) == 1:
        extraction_index = available_indices[0]

        matched_pairs.append({
            "reference_index": reference_index,
            "extraction_index": extraction_index,
        })

        matched_extraction_indices.add(
            extraction_index
        )

    else:
        missing_reference_indices.append(
            reference_index
        )


all_extraction_indices = set(
    extracted_comparison_df["_extraction_index"].astype(int)
)

unsupported_extraction_indices = sorted(
    all_extraction_indices - matched_extraction_indices
)

missing_reference_indices = sorted(
    missing_reference_indices
)

print("Aligned records:", len(matched_pairs))
print("Missing reference records:", len(missing_reference_indices))
print(
    "Unsupported/unmatched extracted records:",
    len(unsupported_extraction_indices),
)


In [ ]:
# ============================================================
# 11. Missing and unsupported/unmatched record tables
# ============================================================
missing_records_df = (
    reference_comparison_df.loc[
        reference_comparison_df["_reference_index"].isin(
            missing_reference_indices
        ),
        FIELDS + ["_reference_index"],
    ]
    .rename(columns={"_reference_index": "Reference Index"})
    .reset_index(drop=True)
)

unsupported_records_df = (
    extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"].isin(
            unsupported_extraction_indices
        ),
        FIELDS + ["_extraction_index"],
    ]
    .rename(columns={"_extraction_index": "Extraction Index"})
    .reset_index(drop=True)
)

print("Missing records:", len(missing_records_df))
print(
    "Unsupported/unmatched records:",
    len(unsupported_records_df),
)


In [ ]:
# ============================================================
# 12. Field-level comparison of aligned records
# ============================================================

comparison_rows = []

for pair in matched_pairs:

    reference_row = reference_comparison_df.loc[
        reference_comparison_df["_reference_index"]
        == pair["reference_index"]
    ].iloc[0]

    extracted_row = extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"]
        == pair["extraction_index"]
    ].iloc[0]

    field_matches = {
        "Category": exact_normalised_text_match(
            reference_row["Category"],
            extracted_row["Category"],
        ),

        "Topic": exact_normalised_text_match(
            reference_row["Topic"],
            extracted_row["Topic"],
        ),

        "Description": exact_normalised_text_match(
            reference_row["Description"],
            extracted_row["Description"],
        ),

        "Value": value_correct(
            reference_row["Value"],
            extracted_row["Value"],
        ),

        "Unit": unit_correct(
            reference_row["Unit"],
            extracted_row["Unit"],
        ),

        "Reporting Period": period_correct(
            reference_row["Reporting Period"],
            extracted_row["Reporting Period"],
        ),

        "Source Location": source_location_correct(
            reference_row["Source Location"],
            extracted_row["Source Location"],
        ),
    }

    all_mismatched_fields = [
        field
        for field in FIELDS
        if not field_matches[field]
    ]

    primary_mismatched_fields = [
        field
        for field in PRIMARY_CORRECTNESS_FIELDS
        if not field_matches[field]
    ]

    fully_correct = (
        len(primary_mismatched_fields) == 0
    )

    output_row = {
        "Reference Index": pair["reference_index"],
        "Extraction Index": pair["extraction_index"],
        "Category": reference_row["Category"],
        "Topic": reference_row["Topic"],
        "Description Lexical Similarity": lexical_similarity(
            reference_row["Description"],
            extracted_row["Description"],
        ),
        "Fully Correct": bool(fully_correct),
        "all_mismatched_fields":
            ", ".join(all_mismatched_fields),
        "primary_mismatched_fields":
            ", ".join(primary_mismatched_fields),
    }

    for field in FIELDS:
        output_row[f"Reference {field}"] = reference_row[field]
        output_row[f"Extracted {field}"] = extracted_row[field]
        output_row[f"{field} Match"] = bool(
            field_matches[field]
        )

    comparison_rows.append(output_row)


comparison_df = pd.DataFrame(comparison_rows)

print("Compared aligned records:", len(comparison_df))
print(
    "Fully correct primary records:",
    int(comparison_df["Fully Correct"].sum())
    if not comparison_df.empty
    else 0,
)

display(comparison_df)


In [ ]:
# ============================================================
# 13. Split fully correct and discrepant records
# ============================================================
if comparison_df.empty:
    fully_correct_records_df = comparison_df.copy()
    discrepant_records_df = comparison_df.copy()
else:
    fully_correct_records_df = comparison_df.loc[
        comparison_df["Fully Correct"]
    ].copy()

    discrepant_records_df = comparison_df.loc[
        ~comparison_df["Fully Correct"]
    ].copy()

print("Fully correct:", len(fully_correct_records_df))
print("Discrepant:", len(discrepant_records_df))

if not discrepant_records_df.empty:
    display(
        discrepant_records_df[
            [
                "Reference Index",
                "Extraction Index",
                "Category",
                "Topic",
                "primary_mismatched_fields",
            ]
        ].reset_index(drop=True)
    )


In [ ]:

# ============================================================
# 14. Field-level accuracy diagnostics
# ============================================================

field_rows = []

for field in FIELDS:

    evaluated_count = len(comparison_df)

    correct_count = (
        int(comparison_df[f"{field} Match"].sum())
        if evaluated_count > 0
        else 0
    )

    accuracy = (
        correct_count / evaluated_count
        if evaluated_count > 0
        else None
    )

    role = (
        "diagnostic"
        if field == DESCRIPTION_DIAGNOSTIC_FIELD
        else "primary"
    )

    field_rows.append({
        "Field": field,
        "Role": role,
        "used_in_alignment_identity":
            field in ALIGNMENT_IDENTITY_FIELDS,
        "used_in_primary_correctness":
            field in PRIMARY_CORRECTNESS_FIELDS,
        "Aligned Records": evaluated_count,
        "Correct Records": correct_count,
        "Incorrect Records":
            evaluated_count - correct_count,
        "Accuracy": accuracy,
        "Overall Accuracy Against Reference": (
            correct_count / len(reference_cmp)
            if len(reference_cmp) > 0
            else None
        ),
    })

field_validation_df = pd.DataFrame(field_rows)

field_error_summary_df = field_validation_df.loc[
    field_validation_df["Incorrect Records"] > 0
].copy()

display(field_validation_df)


In [ ]:

# ============================================================
# 15. Calculate common validation metrics
# ============================================================

reference_record_count = len(reference_cmp)
extracted_record_count = len(extraction_cmp)
aligned_record_count = len(comparison_df)

fully_correct_record_count = (
    int(comparison_df["Fully Correct"].sum())
    if not comparison_df.empty
    else 0
)

discrepant_record_count = (
    aligned_record_count - fully_correct_record_count
)

missing_record_count = len(missing_records_df)
unsupported_record_count = len(unsupported_records_df)

completeness = (
    aligned_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

missing_rate = (
    missing_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

unsupported_rate = (
    unsupported_record_count / extracted_record_count
    if extracted_record_count > 0
    else 0.0
)

record_precision_exact = (
    fully_correct_record_count / extracted_record_count
    if extracted_record_count > 0
    else 0.0
)

record_recall_exact = (
    fully_correct_record_count / reference_record_count
    if reference_record_count > 0
    else 0.0
)

record_f1_exact = (
    2 * record_precision_exact * record_recall_exact
    / (record_precision_exact + record_recall_exact)
    if record_precision_exact + record_recall_exact > 0
    else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_record_count / aligned_record_count
    if aligned_record_count > 0
    else 0.0
)

correct_primary_field_instances = int(
    sum(
        comparison_df[f"{field} Match"].sum()
        for field in PRIMARY_CORRECTNESS_FIELDS
    )
)

expected_primary_field_instances = int(
    reference_record_count
    * len(PRIMARY_CORRECTNESS_FIELDS)
)

field_accuracy = (
    correct_primary_field_instances
    / expected_primary_field_instances
    if expected_primary_field_instances > 0
    else 0.0
)

description_row = field_validation_df.loc[
    field_validation_df["Field"] == DESCRIPTION_DIAGNOSTIC_FIELD
]

description_diagnostic_accuracy = (
    float(description_row.iloc[0]["Accuracy"])
    if (
        not description_row.empty
        and pd.notna(description_row.iloc[0]["Accuracy"])
    )
    else None
)

print("Reference records:", reference_record_count)
print("Extracted records:", extracted_record_count)
print("Aligned records:", aligned_record_count)
print("Fully correct:", fully_correct_record_count)
print("Discrepant:", discrepant_record_count)
print("Missing:", missing_record_count)
print("Unsupported/unmatched:", unsupported_record_count)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print("Field accuracy:", round(field_accuracy, 4))
print(
    "Description diagnostic accuracy:",
    None
    if description_diagnostic_accuracy is None
    else round(description_diagnostic_accuracy, 4),
)


In [ ]:
# ============================================================
# 16. Category-level metrics
# ============================================================

category_rows = []

for category in EXPECTED_CATEGORY_COUNTS:

    expected_records = int(
        (reference_cmp["Category"] == category).sum()
    )

    extracted_records_category = sum(
        1
        for record in extraction_cmp.to_dict('records')
        if record.get("Category") == category
    )

    category_comparison = (
        comparison_df.loc[
            comparison_df["Category"] == category
        ]
        if not comparison_df.empty
        else comparison_df
    )

    aligned_records_category = len(category_comparison)

    fully_correct_category = (
        int(category_comparison["Fully Correct"].sum())
        if not category_comparison.empty
        else 0
    )

    discrepant_category = (
        aligned_records_category - fully_correct_category
    )

    category_completeness = (
        aligned_records_category / expected_records
        if expected_records > 0
        else 0.0
    )

    category_precision = (
        fully_correct_category / extracted_records_category
        if extracted_records_category > 0
        else 0.0
    )

    category_recall = (
        fully_correct_category / expected_records
        if expected_records > 0
        else 0.0
    )

    category_f1 = (
        2 * category_precision * category_recall
        / (category_precision + category_recall)
        if category_precision + category_recall > 0
        else 0.0
    )

    category_rows.append({
        "Category": category,
        "Expected Records": expected_records,
        "Extracted Records": extracted_records_category,
        "Aligned Records": aligned_records_category,
        "Fully Correct Records": fully_correct_category,
        "Discrepant Records": discrepant_category,
        "Completeness": category_completeness,
        "Record Precision Exact": category_precision,
        "Record Recall Exact": category_recall,
        "Record F1 Exact": category_f1,
    })


category_metrics_df = pd.DataFrame(category_rows)
display(category_metrics_df)


In [ ]:
# ============================================================
# 17. D9 corrected-reference integrity confirmation
# ============================================================

reference_semantic_checks = {
    "index_entry_values_null": bool(
        reference_cmp.loc[
            reference_cmp["Category"] == "Index entry",
            "Value"
        ].isna().all()
    ),

    "document_structure_values_null": bool(
        reference_cmp.loc[
            reference_cmp["Category"] == "Document structure",
            "Value"
        ].isna().all()
    ),

    "document_structure_periods_null": bool(
        reference_cmp.loc[
            reference_cmp["Category"] == "Document structure",
            "Reporting Period"
        ].isna().all()
    ),

    "portugal_area_period_null": bool(
        pd.isna(
            reference_cmp.loc[
                reference_cmp["Topic"] == "Portugal area",
                "Reporting Period"
            ].iloc[0]
        )
    ),
}

reference_semantics_valid = all(
    reference_semantic_checks.values()
)

print(
    json.dumps(
        reference_semantic_checks,
        indent=2,
        ensure_ascii=False,
    )
)

print(
    "Corrected D9 reference semantics valid:",
    reference_semantics_valid,
)

if not reference_semantics_valid:
    raise AssertionError(
        "The supplied D9 reference does not match the "
        "corrected frozen Stage 1 reference semantics."
    )


In [ ]:

# ============================================================
# 18. Build final Branch B validation summary
# ============================================================

field_accuracy_dictionary = {
    row["Field"]: (
        None
        if pd.isna(row["Accuracy"])
        else float(row["Accuracy"])
    )
    for _, row in field_validation_df.iterrows()
}

category_metrics_dictionary = {
    row["Category"]: {
        "expected_records": int(row["Expected Records"]),
        "extracted_records": int(row["Extracted Records"]),
        "aligned_records": int(row["Aligned Records"]),
        "fully_correct_records":
            int(row["Fully Correct Records"]),
        "discrepant_records":
            int(row["Discrepant Records"]),
        "completeness": float(row["Completeness"]),
        "record_precision_exact":
            float(row["Record Precision Exact"]),
        "record_recall_exact":
            float(row["Record Recall Exact"]),
        "record_f1_exact":
            float(row["Record F1 Exact"]),
    }
    for _, row in category_metrics_df.iterrows()
}

summary = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,

    "reference_records": int(reference_record_count),
    "extracted_records": int(extracted_record_count),
    "aligned_records": int(aligned_record_count),
    "fully_correct_records": int(fully_correct_record_count),
    "discrepant_records": int(discrepant_record_count),
    "missing_records": int(missing_record_count),
    "unsupported_extracted_records": int(unsupported_record_count),

    "completeness": round(float(completeness), 4),
    "missing_rate": round(float(missing_rate), 4),
    "record_precision_exact":
        round(float(record_precision_exact), 4),
    "record_recall_exact":
        round(float(record_recall_exact), 4),
    "record_f1_exact":
        round(float(record_f1_exact), 4),
    "unsupported_rate":
        round(float(unsupported_rate), 4),
    "discrepancy_rate_among_aligned":
        round(float(discrepancy_rate_among_aligned), 4),
    "field_accuracy":
        round(float(field_accuracy), 4),

    "description_diagnostic_accuracy": (
        None
        if description_diagnostic_accuracy is None
        else round(float(description_diagnostic_accuracy), 4)
    ),

    "field_accuracy_among_aligned":
        field_accuracy_dictionary,

    "schema_validity": bool(schema_validity),
    "schema_diagnostics": schema_diagnostics,
    "structurally_evaluable": structurally_evaluable,
    "content_diagnostics": content_diagnostics,

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "matching_rules": {
        "identity_fields":
            ALIGNMENT_IDENTITY_FIELDS,
        "one_to_one_assignment":
            "Unique deterministic Category + Topic identity",
        "value_used_for_alignment": False,
        "unit_used_for_alignment": False,
        "reporting_period_used_for_alignment": False,
        "description_used_for_alignment": False,
        "source_location_used_for_alignment": False,
    },

    "comparison_rules_frozen_from_branch_A": True,

    "comparison_rules": {
        "raw_extraction_modified": False,
        "manual_correction_applied": False,
        "comparison_normalisation_scope":
            "Comparison copies only",
        "numeric_comparison":
            "Exact represented numeric equality after deterministic parsing",
        "text_value_comparison":
            "Normalised exact textual equality; no fuzzy correctness",
        "description":
            (
                "Exact-string diagnostic plus lexical similarity diagnostic; "
                "excluded from primary exact-record correctness because the "
                "task permits a concise source-grounded description"
            ),
        "unit":
            "Controlled notation equivalence only",
        "reporting_period":
            "Normalised exact correctness with dash-notation normalisation only",
        "source_location":
            (
                "Normalised physical/source provenance; 'Physical PDF' prefix "
                "and Índice/Index terminology treated as equivalent"
            ),
        "d9_equivalence_rules_status":
            (
                "Final D9 Validation A document/schema-level comparison rules "
                "reused unchanged. No Branch-B-specific performance-driven "
                "equivalence rules were added."
            ),
    },

    "reference_integrity_confirmation": {
        "reference_semantics_valid":
            bool(reference_semantics_valid),
        "checks":
            reference_semantic_checks,
        "reference_modified_by_validation":
            False,
    },

    "category_metrics":
        category_metrics_dictionary,

    "input_provenance":
        input_provenance,
}

print(json.dumps(
    summary,
    ensure_ascii=False,
    indent=2,
))


In [ ]:

# ============================================================
# 19. Validation integrity checks
# ============================================================

assert (
    aligned_record_count
    + missing_record_count
    == reference_record_count
)

assert (
    aligned_record_count
    + unsupported_record_count
    == extracted_record_count
)

assert (
    fully_correct_record_count
    + discrepant_record_count
    == aligned_record_count
)

assert reference_semantics_valid

for metric_name, metric_value in {
    "completeness": completeness,
    "missing_rate": missing_rate,
    "record_precision_exact": record_precision_exact,
    "record_recall_exact": record_recall_exact,
    "record_f1_exact": record_f1_exact,
    "unsupported_rate": unsupported_rate,
    "discrepancy_rate_among_aligned":
        discrepancy_rate_among_aligned,
    "field_accuracy": field_accuracy,
}.items():

    assert 0.0 <= metric_value <= 1.0, (
        f"Invalid {metric_name}: {metric_value}"
    )

print("Validation integrity checks passed.")


In [ ]:

# ============================================================
# 20. Export validation artefacts
# ============================================================

DETAILED_PATH = (
    OUTPUT_DIR / "D9_branch_B_validation_detailed.csv"
)

FULLY_CORRECT_PATH = (
    OUTPUT_DIR / "D9_branch_B_fully_correct_records.csv"
)

DISCREPANT_PATH = (
    OUTPUT_DIR / "D9_branch_B_discrepant_records.csv"
)

MISSING_PATH = (
    OUTPUT_DIR / "D9_branch_B_missing_records.csv"
)

UNSUPPORTED_PATH = (
    OUTPUT_DIR / "D9_branch_B_unsupported_records.csv"
)

FIELD_VALIDATION_PATH = (
    OUTPUT_DIR / "D9_branch_B_field_validation.csv"
)

FIELD_ERROR_SUMMARY_PATH = (
    OUTPUT_DIR / "D9_branch_B_field_error_summary.csv"
)

CATEGORY_METRICS_PATH = (
    OUTPUT_DIR / "D9_branch_B_category_metrics.csv"
)

SUMMARY_PATH = (
    OUTPUT_DIR / "D9_branch_B_validation_summary.json"
)

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig",
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_PATH,
    index=False,
    encoding="utf-8-sig",
)

discrepant_records_df.to_csv(
    DISCREPANT_PATH,
    index=False,
    encoding="utf-8-sig",
)

missing_records_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig",
)

unsupported_records_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig",
)

field_validation_df.to_csv(
    FIELD_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

field_error_summary_df.to_csv(
    FIELD_ERROR_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig",
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

print("D9 Validation B artefacts saved.")


In [ ]:

# ============================================================
# 21. Download generated validation artefacts
# ============================================================

GENERATED_OUTPUTS = [
    DETAILED_PATH,
    FULLY_CORRECT_PATH,
    DISCREPANT_PATH,
    MISSING_PATH,
    UNSUPPORTED_PATH,
    FIELD_VALIDATION_PATH,
    FIELD_ERROR_SUMMARY_PATH,
    CATEGORY_METRICS_PATH,
    SUMMARY_PATH,
]

for output_path in GENERATED_OUTPUTS:
    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists(),
    )

for output_path in GENERATED_OUTPUTS:
    if output_path.exists():
        files.download(output_path)
